# Quito, a Weekday in One Assignment

A whole weekday as one O-D matrix, assigned once against capacity scaled up to a day.

Reads what `TAZ_OD_demo.ipynb` exported and rebuilds none of it:

| needs | from |
|---|---|
| 761 zones | `outputs/taz_od_exports/zones.gpkg` |
| O-D matrix | `outputs/taz_od_exports/od_matrix.omx` |
| network and connectors | `data/road_model/` |

The exports are committed. `data/road_model/` is not: it is written by a run of
`TAZ_OD_demo.ipynb`, which needs `data/gov_data/`.

In [ ]:
import os
import shutil

import folium
import geopandas as gpd
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)

_cwd = os.getcwd()
BASE_DIR = _cwd if os.path.isdir(os.path.join(_cwd, "outputs")) else os.path.dirname(_cwd)
EXPORTS = os.path.join(BASE_DIR, "outputs", "taz_od_exports")
MAPS_DIR = os.path.join(BASE_DIR, "outputs", "weekday_maps")
os.makedirs(MAPS_DIR, exist_ok=True)
for _stale in os.listdir(MAPS_DIR):
    _path = os.path.join(MAPS_DIR, _stale)
    if os.path.isfile(_path):
        os.remove(_path)

SOURCE_MODEL = os.path.join(BASE_DIR, "data", "road_model")
WORK_MODEL = os.environ.get("WEEKDAY_MODEL_DIR",
                            os.path.join(BASE_DIR, "data", "weekday_model"))
METRE_CRS = "EPSG:32717"

for _needed in (os.path.join(EXPORTS, "zones.gpkg"),
                os.path.join(EXPORTS, "od_matrix.omx"),
                os.path.join(SOURCE_MODEL, "project_database.sqlite")):
    if not os.path.exists(_needed):
        raise FileNotFoundError(f"Missing input: {_needed}\n"
                                f"Run TAZ_OD_demo.ipynb first - it writes these.")

print(f"exports: {EXPORTS}\nmodel:   {WORK_MODEL}\nmaps:    {MAPS_DIR}")

## A1. The weekday matrix

The road model's morning peak hour, `T`, runs from home to activity. Over a whole day every trip
out has a trip home, so the day's matrix is symmetric:

`day = WEEKDAY_SIZE × (T + T') / 2`

`WEEKDAY_SIZE` is the weekday's traffic in morning-peak hours. It is not measured.

In [ ]:
from aequilibrae.matrix import AequilibraeMatrix                       # noqa: E402

zones = gpd.read_file(os.path.join(EXPORTS, "zones.gpkg"))

_read = AequilibraeMatrix()
_read.create_from_omx(os.path.join(EXPORTS, "od_matrix.omx"),
                      os.path.join(WORK_MODEL + "_tmp.aem"), memory_only=True)
HOME_TO_ACTIVITY = np.array(_read.matrix["car"], dtype=float)
ZONE_IDS = np.array(_read.index[:], dtype=np.int64)
_read.close()
if os.path.exists(WORK_MODEL + "_tmp.aem"):
    os.remove(WORK_MODEL + "_tmp.aem")
print(f"{len(zones):,} zones; morning peak hour: {HOME_TO_ACTIVITY.sum():,.0f} vehicle trips")

# A weekday's traffic in morning-peak hours.
WEEKDAY_SIZE = 10.16

DAY_MATRIX = WEEKDAY_SIZE * (HOME_TO_ACTIVITY + HOME_TO_ACTIVITY.T) / 2
print(f"weekday: {DAY_MATRIX.sum():,.0f} vehicle trips, "
      f"symmetric: {np.allclose(DAY_MATRIX, DAY_MATRIX.T)}")

## A2. Daily capacity

A link's capacity is vehicles per hour. Multiplied by `F` it becomes vehicles per day, so the
daily matrix can be assigned against it:

`F = 1 / BUSIEST_HOUR_SHARE`

On a weekday the busiest hour is the morning peak, so its share of the day is `1 / WEEKDAY_SIZE`.
At that `F` the day is assigned as if its busiest hour were repeated all day long.

In [ ]:
BUSIEST_HOUR_SHARE = 1 / WEEKDAY_SIZE
CAPACITY_FACTOR = 1 / BUSIEST_HOUR_SHARE
print(f"the busiest hour is {100 * BUSIEST_HOUR_SHARE:.1f}% of the day, so F = "
      f"{CAPACITY_FACTOR:.2f}")

## A3. The road model

A copy, so the run writing results leaves `data/road_model/` untouched. The daily capacity is one
more column on the graph; the network itself is not changed.

In [ ]:
import logging                                                        # noqa: E402

from aequilibrae import Project                                       # noqa: E402
from aequilibrae.paths import TrafficAssignment, TrafficClass         # noqa: E402


def _release_log_files(folder):
    """AequilibraE keeps a log file open inside the project; Windows will not delete it."""
    target = os.path.abspath(folder)
    for logger in [logging.getLogger()] + [logging.getLogger(n) for n
                                           in list(logging.Logger.manager.loggerDict)]:
        for handler in list(getattr(logger, "handlers", [])):
            written = getattr(handler, "baseFilename", None)
            if written and os.path.abspath(written).startswith(target):
                handler.close()
                logger.removeHandler(handler)


def open_working_copy():
    """Copy the saved model and open the copy."""
    if os.path.isdir(WORK_MODEL):
        _release_log_files(WORK_MODEL)
        try:
            shutil.rmtree(WORK_MODEL)
        except PermissionError as broken:
            raise PermissionError(
                f"{WORK_MODEL} is held open by another process, so it cannot be replaced. "
                f"Close any notebook using it, or set WEEKDAY_MODEL_DIR to another path."
            ) from broken
    shutil.copytree(SOURCE_MODEL, WORK_MODEL,
                    ignore=shutil.ignore_patterns("*.log", "osm_data"))
    project = Project()
    project.open(WORK_MODEL)
    return project


model = open_working_copy()
model.network.build_graphs(modes=["c"])
car_graph = model.network.graphs["c"]
car_graph.prepare_graph(ZONE_IDS)
car_graph.set_graph("travel_time")
car_graph.set_skimming(["travel_time"])
car_graph.set_blocked_centroid_flows(True)
car_graph.graph["capacity_daily"] = car_graph.graph["capacity"] * CAPACITY_FACTOR
print(f"graph ready: {car_graph.compact_num_links:,} compact links, "
      f"{len(ZONE_IDS):,} centroids")

## B1. The assignment

User equilibrium with BPR, against the `capacity_daily` column named in `set_capacity_field()`.
The flows that come out are vehicles per day.

In [ ]:
BPR_ALPHA, BPR_BETA = 0.15, 4.0

RGAP_TARGET, MAX_ITER = 0.001, 200


def as_matrix(array, name="car"):
    """Wrap a zone-by-zone array as the demand matrix the solver reads."""
    holder = AequilibraeMatrix()
    holder.create_empty(zones=array.shape[0], matrix_names=[name], memory_only=True)
    holder.index[:] = ZONE_IDS
    holder.matrices[:, :, 0] = array
    holder.computational_view([name])
    return holder


job = TrafficAssignment()
job.add_class(TrafficClass(name="car", graph=car_graph, matrix=as_matrix(DAY_MATRIX)))
job.set_vdf("BPR")
job.set_vdf_parameters({"alpha": BPR_ALPHA, "beta": BPR_BETA})
job.set_capacity_field("capacity_daily")
job.set_time_field("travel_time")
job.set_algorithm("bfw")
job.max_iter = MAX_ITER
job.rgap_target = RGAP_TARGET
job.execute()
job.save_results(table_name="weekday")
print(f"gap {job.assignment.rgap:.2e} after {job.assignment.iter} iterations"
      + ("" if job.assignment.rgap <= RGAP_TARGET else "   ** did not reach the target"))
print(f"results saved in {os.path.join(WORK_MODEL, 'results_database.sqlite')}")

## B2. The weekday

V/C is daily volume over daily capacity: the same ratio the busiest hour would have.

In [ ]:
CAR_AB, CAR_BA = "car_ab", "car_ba"

with model.db_connection as _conn:
    links = pd.read_sql(
        "SELECT link_id, name, link_type, direction, distance, capacity_ab, capacity_ba, "
        "travel_time_ab, travel_time_ba, lanes_ab, speed_ab FROM links", _conn)

loaded = job.results().rename_axis("link_id").reset_index().merge(links, on="link_id", how="left")
loaded = loaded[loaded["link_type"] != "centroid_connector"].set_index("link_id")
for _col in (CAR_AB, CAR_BA, "Congested_Time_AB", "Congested_Time_BA",
             "travel_time_ab", "travel_time_ba"):
    loaded[_col] = loaded[_col].fillna(0)
for _side, _cap in (("ab", "capacity_ab"), ("ba", "capacity_ba")):
    loaded[f"voc_{_side}"] = (loaded[f"car_{_side}"]
                              / (CAPACITY_FACTOR * loaded[_cap].replace(0, np.nan)))
loaded["voc"] = loaded[["voc_ab", "voc_ba"]].max(axis=1).fillna(0)
loaded["daily"] = loaded[CAR_AB] + loaded[CAR_BA]
loaded["veh_h"] = (loaded[CAR_AB] * loaded["Congested_Time_AB"]
                   + loaded[CAR_BA] * loaded["Congested_Time_BA"]) / 60
loaded["free_h"] = (loaded[CAR_AB] * loaded["travel_time_ab"]
                    + loaded[CAR_BA] * loaded["travel_time_ba"]) / 60

totals = {"vehicle trips": DAY_MATRIX.sum(),
          "veh-km (thousands)": (loaded["daily"] * loaded["distance"]).sum() / 1e6,
          "veh-hours": loaded["veh_h"].sum(),
          "delay %": 100 * (loaded["veh_h"].sum() / loaded["free_h"].sum() - 1),
          "km over capacity": loaded.loc[loaded["voc"] >= 1, "distance"].sum() / 1000,
          "worst V/C": loaded["voc"].max()}
print(pd.Series(totals).to_string(float_format=lambda v: f"{v:,.2f}"))

## B3. The weekday on a map

Coloured by daily V/C. The hover gives each street's daily flow.

In [ ]:
import shapely                                                        # noqa: E402

# Bands, colours and labels exactly as the road notebook draws them.
VOC_BANDS = [(0.00, 0.50, "#2c7bb6", "under 50% - free flowing"),
             (0.50, 0.85, "#8cc4de", "50-85% - busy"),
             (0.85, 1.00, "#fdae61", "85-100% - at capacity"),
             (1.00, 1.50, "#d7191c", "100-150% - over capacity"),
             (1.50, 1e9, "#7d0018", "over 150% - far beyond capacity")]
LINE_WEIGHT = {"motorway": 3.6, "trunk": 3.2, "primary": 2.6, "secondary": 2.0, "tertiary": 1.4}
MAP_ROAD_TYPES = ["motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link",
                  "secondary", "secondary_link", "tertiary", "tertiary_link"]
TIP_CSS = ("font-family:sans-serif;font-size:13px;line-height:1.45;max-width:285px;"
           "background:white;padding:9px 12px;border:1px solid #999;border-radius:6px;")
BOX = ("position:fixed;bottom:24px;left:24px;z-index:9999;background:white;"
       "padding:9px 12px;border:1px solid #999;border-radius:6px;"
       "font-family:sans-serif;font-size:13px;line-height:1.45;max-width:285px;")
PANEL = ("position:fixed;top:20px;right:20px;z-index:9999;background:white;"
         "padding:9px 12px;border:1px solid #999;border-radius:6px;"
         "font-family:sans-serif;font-size:13px;line-height:1.5;max-width:265px;")
# Esri's Light Gray Canvas: a pale base with its place labels as a separate layer. It needs no API
# key, only the attribution shown in the corner.
ESRI_CANVAS = ("https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/{}/MapServer/tile/"
               "{{z}}/{{y}}/{{x}}")
ESRI_ATTRIBUTION = "Tiles &copy; Esri &mdash; Esri, HERE, Garmin, &copy; OpenStreetMap contributors"


def swatches():
    """The legend rows, drawn as the road notebook draws them."""
    return "".join(
        f'<div><span style="display:inline-block;width:16px;height:4px;background:{c};'
        f'margin:0 6px 3px 0;vertical-align:middle;"></span>{label}</div>'
        for _, _, c, label in VOC_BANDS)


def add_basemap(m):
    """Pale grey base and its labels, kept out of the layer control."""
    for layer, name in (("World_Light_Gray_Base", "Light grey"),
                        ("World_Light_Gray_Reference", "Place names")):
        folium.TileLayer(tiles=ESRI_CANVAS.format(layer), attr=ESRI_ATTRIBUTION, name=name,
                         max_zoom=16, control=False).add_to(m)
    return m


def band_colour(voc):
    for lo, hi, colour, _ in VOC_BANDS:
        if lo <= voc < hi:
            return colour
    return VOC_BANDS[-1][2]


with model.db_connection_spatial as _conn:
    _geo = pd.read_sql("SELECT link_id, AsBinary(geometry) wkb FROM links", _conn)
_geo["geometry"] = [shapely.wkb.loads(bytes(b)) for b in _geo["wkb"]]
_geo = gpd.GeoDataFrame(_geo.drop(columns="wkb"), geometry="geometry", crs="EPSG:4326")
_geo["geometry"] = _geo.to_crs(METRE_CRS).simplify(8).to_crs("EPSG:4326")
_geo["geometry"] = shapely.set_precision(_geo.geometry.values, 1e-5)

_d = loaded[(loaded["daily"] > 0) & loaded["link_type"].isin(MAP_ROAD_TYPES)].reset_index()
drawn = gpd.GeoDataFrame(_d.merge(_geo, on="link_id", how="inner"),
                         geometry="geometry", crs="EPSG:4326")
_base = drawn["link_type"].str.replace("_link", "", regex=False)
drawn["colour"] = [band_colour(v) for v in drawn["voc"]]
drawn["weight"] = (_base.map(LINE_WEIGHT).fillna(1.4)
                   * np.where(drawn["link_type"].str.endswith("_link"), 0.6, 1.0))
drawn["street"] = drawn["name"].fillna("unnamed").replace("", "unnamed")
drawn["class"] = drawn["link_type"]
drawn["lanes"] = drawn["lanes_ab"].fillna(0).astype(int)
drawn["capacity veh/day"] = (CAPACITY_FACTOR * drawn["capacity_ab"].fillna(0)).round(0)
drawn["A to B veh/day"] = drawn[CAR_AB].round(0)
drawn["B to A veh/day"] = drawn[CAR_BA].round(0)
drawn["V/C"] = drawn["voc"].round(2)
drawn["metres"] = drawn["distance"].round(0)
fields = ["street", "class", "lanes", "capacity veh/day", "metres",
          "A to B veh/day", "B to A veh/day", "V/C", "link_id"]
aliases = ["", "class", "lanes", "capacity veh/day", "metres",
           "A&rarr;B veh/day", "B&rarr;A veh/day", "V/C", "link"]

_centre = zones.to_crs("EPSG:4326").geometry.union_all().centroid
m = folium.Map(location=[_centre.y, _centre.x], zoom_start=11, tiles=None, prefer_canvas=True)
add_basemap(m)
folium.GeoJson(
    drawn[fields + ["colour", "weight", "geometry"]],
    name="weekday",
    style_function=lambda feature: {"color": feature["properties"]["colour"],
                                    "weight": feature["properties"]["weight"],
                                    "opacity": 0.85},
    highlight_function=lambda _feature: {"weight": 7, "opacity": 1},
    tooltip=folium.GeoJsonTooltip(fields=fields, aliases=aliases, sticky=False, style=TIP_CSS),
).add_to(m)
m.get_root().html.add_child(folium.Element(
    f'<div style="{BOX}"><b>Volume over capacity, daily</b>{swatches()}</div>'))
_over_km = drawn[drawn["voc"] >= 1.0]["distance"].sum() / 1000
m.get_root().html.add_child(folium.Element(
    f'<div style="{PANEL}"><b>weekday, one assignment</b>'
    f'<div>{totals["vehicle trips"]:,.0f} vehicle trips over {len(zones):,} zones</div>'
    f'<div>{totals["veh-hours"]:,.0f} vehicle-hours, {totals["delay %"]:.0f}% more '
    f'than the same trips on empty streets</div>'
    f'<div>{_over_km:,.0f} of {drawn["distance"].sum() / 1000:,.0f} km drawn are over capacity'
    f'</div></div>'))
_file = os.path.join(MAPS_DIR, "weekday.html")
m.save(_file)
print(f"{len(drawn):,} links drawn   weekday.html   {os.path.getsize(_file) / 1e6:.1f} MB")

## B4. The matrix written out

The day's O-D matrix, in two formats:

| file | what it holds |
|---|---|
| `weekday_matrix.omx` | the matrix, square, in the format AequilibraE reads back |
| `weekday_matrix.csv.gz` | the same matrix, one row per zone pair that carries trips |

In [ ]:
import tempfile                                                       # noqa: E402

EXPORT_DIR = os.path.join(BASE_DIR, "outputs", "weekday_exports")
os.makedirs(EXPORT_DIR, exist_ok=True)
for _old in os.listdir(EXPORT_DIR):
    _path = os.path.join(EXPORT_DIR, _old)
    if os.path.isfile(_path):
        os.remove(_path)

MIN_TRIPS = 0.01          # a pair below this is rounding noise, not demand

_omx_path = os.path.join(EXPORT_DIR, "weekday_matrix.omx")
_out = AequilibraeMatrix()
_out.create_empty(zones=len(ZONE_IDS), matrix_names=["car"], index_names=["taz"],
                  memory_only=True)
_out.index[:] = ZONE_IDS
_out.matrices[:, :, 0] = DAY_MATRIX
_out.computational_view(["car"])
_out.export(_omx_path)
_out.close()

# Read it back, so a matrix that does not reopen is caught here.
_check = AequilibraeMatrix()
_check.create_from_omx(_omx_path, os.path.join(tempfile.gettempdir(), "_weekday_matrix_check.aem"),
                       memory_only=True)
if not np.array_equal(_check.index[:], ZONE_IDS):
    raise ValueError("the exported matrix does not carry the zone numbers")
_worst_cell = float(np.abs(np.array(_check.matrix["car"], dtype=float) - DAY_MATRIX).max())
_check.close()
print(f"  {'weekday_matrix.omx':<26}{os.path.getsize(_omx_path) / 1e6:>6.2f} MB   reopened, largest "
      f"difference {_worst_cell:.2e}")

_origin, _dest = np.meshgrid(ZONE_IDS, ZONE_IDS, indexing="ij")
_carries = DAY_MATRIX > MIN_TRIPS
_pairs = pd.DataFrame({"origin": _origin[_carries], "destination": _dest[_carries],
                       "trips": DAY_MATRIX[_carries].round(3)})
_csv_path = os.path.join(EXPORT_DIR, "weekday_matrix.csv.gz")
_pairs.to_csv(_csv_path, index=False, compression="gzip")
print(f"  {'weekday_matrix.csv.gz':<26}{os.path.getsize(_csv_path) / 1e6:>6.2f} MB   "
      f"{len(_pairs):,} pairs holding {100 * _pairs['trips'].sum() / DAY_MATRIX.sum():.2f}% "
      f"of the trips")

## B5. What is assumed

Network, zones and matrix come from the road model unchanged. This notebook adds:

| | |
|---|---|
| `WEEKDAY_SIZE` | 10.16 morning-peak hours of traffic in a weekday |
| `F` | 1 / busiest-hour share = `WEEKDAY_SIZE` |
| `BPR_ALPHA, BPR_BETA` | 0.15 and 4.0, BPR's published values for one hour |

### What this does not capture

- **No direction.** The day's matrix is symmetric, so neither peak's one-way loading appears.
- **Congestion is the same all day.** Every hour is the busiest hour. Delay depends on `F` more
  than on anything else.
- **No hours.** No busiest hour per street and no hours over capacity.
- **The day's size is not measured.** One count station with an hourly profile would set it.
- **BPR is set for one hour.** A daily model would fit its own `alpha` and `beta` to observed
  daily speeds.